In [ ]:
# %% [Cell 1: 環境載入與設定]
import os
import numpy as np
import pandas as pd
import sys
import subprocess
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])
    import matplotlib.pyplot as plt

print("Dataset Audit Initialized for Drone IR Vehicle Detection (YouTube Source)")

# %% [Cell 2: 建立/模擬讀取稽核用資料表]
np.random.seed(42)
n_samples = 3500

# 16 支 YouTube 視訊片段分組
videos = [f"yt_clip_{i:02d}" for i in range(1, 17)]
video_assignments = np.random.choice(videos, size=n_samples, p=[0.0625]*16)

# 單一類別: 0 代表 vehicle
classes = np.zeros(n_samples, dtype=int)

# 高空微小車輛長寬 (歸一化 0~1)
widths = np.clip(np.random.exponential(scale=0.035, size=n_samples), 0.006, 0.18)
heights = widths * np.random.uniform(0.6, 1.2, size=n_samples)

# 注入 8 筆 YouTube 視訊壓縮退化邊界框 (異常品質)
widths[5] = 0.001
widths[22] = 0.0008
widths[105] = 0.0005
widths[340] = 0.0003
widths[890] = 0.0004
widths[1200] = 0.0002
widths[2100] = 0.0005
widths[3100] = 0.0003

df = pd.DataFrame({
    'frame_id': [f"frame_{i:05d}" for i in range(n_samples)],
    'video_id': video_assignments,
    'class_id': classes,
    'class_name': ['vehicle'] * n_samples,
    'bbox_width': widths,
    'bbox_height': heights,
    'has_watermark': np.random.choice([0, 1], size=n_samples, p=[0.82, 0.18])
})

print(f"Dataset Shape: {df.shape}")
df.head()

# %% [Cell 3: 基礎品質檢查 (Missing Values & Degenerate Boxes)]
print("=== 1. Missing Values Audit ===")
print(df.isnull().sum())

# 換算為 640x512 解析度下的實際像素長寬
df['bbox_w_px'] = df['bbox_width'] * 640
df['bbox_h_px'] = df['bbox_height'] * 512
df['bbox_area_px'] = df['bbox_w_px'] * df['bbox_h_px']

degenerate_boxes = df[df['bbox_w_px'] <= 2.0]
print(f"\n=== 2. Degenerate Boxes Count (w <= 2px): {len(degenerate_boxes)} ===")
display(degenerate_boxes[['frame_id', 'video_id', 'bbox_w_px', 'bbox_h_px']])

# %% [Cell 4: 目標尺度分佈 (Scale Distribution Audit)]
print("=== 3. Target Scale Distribution Audit ===")
tiny_vehicles = df[df['bbox_area_px'] < (16 * 16)]
small_vehicles = df[(df['bbox_area_px'] >= (16 * 16)) & (df['bbox_area_px'] < (32 * 32))]
large_vehicles = df[df['bbox_area_px'] >= (32 * 32)]

print(f"Tiny Vehicles (< 16x16 px): {len(tiny_vehicles)} ({len(tiny_vehicles)/len(df)*100:.1f}%)")
print(f"Small Vehicles (16x16 ~ 32x32 px): {len(small_vehicles)} ({len(small_vehicles)/len(df)*100:.1f}%)")
print(f"Large Vehicles (> 32x32 px): {len(large_vehicles)} ({len(large_vehicles)/len(df)*100:.1f}%)")

# %% [Cell 5: 切分驗證 (Group Split Audit by video_id)]
train_videos = [f"yt_clip_{i:02d}" for i in range(1, 11)]
val_videos = ['yt_clip_11', 'yt_clip_12', 'yt_clip_13']
test_videos = ['yt_clip_14', 'yt_clip_15', 'yt_clip_16']

df['split'] = 'train'
df.loc[df['video_id'].isin(val_videos), 'split'] = 'val'
df.loc[df['video_id'].isin(test_videos), 'split'] = 'test'

print("=== 4. Split Distribution ===")
print(df['split'].value_counts())

# 驗證 Group Overlap (必須為 0)
train_groups = set(df[df['split'] == 'train']['video_id'])
test_groups = set(df[df['split'] == 'test']['video_id'])
overlap = train_groups.intersection(test_groups)

print(f"\nVideo Group Overlap between Train and Test: {len(overlap)} (Must be 0)")
assert len(overlap) == 0, "Warning: Split Leakage detected!"
print("Group Split Audit Passed: Zero leakage between YouTube video clips.")

Dataset Audit Initialized for Drone IR Vehicle Detection (YouTube Source)
Dataset Shape: (3500, 7)
=== 1. Missing Values Audit ===
frame_id         0
video_id         0
class_id         0
class_name       0
bbox_width       0
bbox_height      0
has_watermark    0
dtype: int64

=== 2. Degenerate Boxes Count (w <= 2px): 8 ===


,frame_id,video_id,bbox_w_px,bbox_h_px
5,frame_00005,yt_clip_03,0.640,14.426432
22,frame_00022,yt_clip_05,0.512,6.589351
105,frame_00105,yt_clip_04,0.320,2.316457
340,frame_00340,yt_clip_08,0.192,15.851768
890,frame_00890,yt_clip_12,0.256,2.534725
1200,frame_01200,yt_clip_13,0.128,3.684382
2100,frame_02100,yt_clip_15,0.320,5.291137
3100,frame_03100,yt_clip_02,0.192,16.167074


=== 3. Target Scale Distribution Audit ===
Tiny Vehicles (< 16x16 px): 2056 (58.7%)
Small Vehicles (16x16 ~ 32x32 px): 816 (23.3%)
Large Vehicles (> 32x32 px): 628 (17.9%)
=== 4. Split Distribution ===
split
train    2170
val       683
test      647
Name: count, dtype: int64

Video Group Overlap between Train and Test: 0 (Must be 0)
Group Split Audit Passed: Zero leakage between YouTube video clips.
